In [1]:
import os
import sys
import scanpy as sc
import scvi
import mudata as md
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
print("Last run with scvi-tools version:", scvi.__version__)
scvi.settings.num_threads = 24
scvi.settings.seed = 0

sc._settings.ScanpyConfig.n_jobs=4
sc.settings.verbosity = 1

Seed set to 0


scanpy==1.10.3 anndata==0.10.8 umap==0.5.6 numpy==1.26.4 scipy==1.14.1 pandas==2.2.2 scikit-learn==1.5.2 statsmodels==0.14.3 igraph==0.11.6 pynndescent==0.5.13
Last run with scvi-tools version: 1.2.0


In [ ]:
scvi.settings.dl_num_workers=2
sc.logging.print_header()

In [ ]:
sc.logging.print_header()

# 1. Run on NVIDIA A100

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "7"
#os.environ["CUDA_VISIBLE_DEVICES"] = "6"

In [ ]:
obj_path = '/home/liyanguo/MyImmuCell/04_MyImmuCell_TOTALVI/'

In [ ]:
dataset = sys.argv[1] #high_nCount_RNA, low_nCount_RNA

In [ ]:
adata = sc.read_h5ad(f"{obj_path}scRNA_MyImmuCell_{dataset}_HVG.h5ad")
adt = sc.read_h5ad(f"{obj_path}scADT_MyImmuCell_{dataset}.h5ad")

In [ ]:
mdata = md.MuData({"rna": adata, "protein": adt})
mdata.update()

In [ ]:
print(f"Max value of protein counts that store in X: {mdata.mod['protein'].X.max()}")

In [ ]:
print(f"Max value of protein counts that store in layers-counts: {mdata.mod['rna'].layers['counts'].max()}")

In [ ]:
scvi.model.TOTALVI.setup_mudata(
    mdata,
    rna_layer="counts",
    protein_layer=None,
    batch_key="Batch",
    modalities={
        "rna_layer": "rna",
        "protein_layer": "protein",
        "batch_key": "protein",
    },
)

In [ ]:
del adata

In [ ]:
model = scvi.model.TOTALVI(mdata,n_latent=30,
                           n_hidden=256, n_layers_decoder=2)
del adata
model.train(accelerator="gpu",lr=0.004,batch_size=4096,
            max_epochs=200)

In [ ]:
model.save(obj_path, overwrite=True, prefix=f'{dataset}_CITEseq_TOTALVI_')

# 2. Get_latent_representation

In [ ]:
X_totalVI = model.get_latent_representation()

In [ ]:
np.save(f"{obj_path}{dataset}_X_TOTALVI",X_totalVI)

# 3. Get model parameter

In [ ]:
sheet=pd.DataFrame()
for key in model.history.keys():
    temp = model.history[key]
    temp = temp.reset_index()
    sheet=pd.concat([sheet,temp],axis=1)

In [ ]:
pd.DataFrame.to_csv(sheet,f"{obj_path}{dataset}_TOTALVI_model_history.csv")

In [ ]:
fig, ax = plt.subplots(1, 1)
sheet["elbo_train"].plot(ax=ax, label="train")
sheet["elbo_validation"].plot(ax=ax, label="validation")
ax.set(title="Negative ELBO over training epochs", ylim=(0, 1400))
ax.legend()
plt.savefig(f"{obj_path}{dataset}_ELBO.png")

In [ ]:
fig, ax = plt.subplots(1, 1)
sheet["train_loss_epoch"].plot(ax=ax, label="train")
sheet["validation_loss"].plot(ax=ax, label="validation")
ax.set(title="loss over training epochs", ylim=(0, 1400))
ax.legend()
plt.savefig(f"{obj_path}{dataset}_loss.png")

# 4. Denoise protein

In [ ]:
rna_denoised, protein_denoised = model.get_normalized_expression()

In [ ]:
adt.layers["denoised_protein"] = protein_denoised

In [ ]:
adt.layers["protein_foreground_prob"] = 100 * model.get_protein_foreground_probability()

In [ ]:
adt.write(f"{obj_path}scADT_MyImmuCell_{dataset}.h5ad",compression="gzip")

In [ ]:
# Shell run on GPU HPC
# nohup python 08B_CITEseq_TOTALVI_Train_high.py high_nCount_RNA > high_nCount_RNA_totalVI.log 2>&1 &
# nohup python 08B_CITEseq_TOTALVI_Train_low.py low_nCount_RNA > low_nCount_RNA_totalVI.log 2>&1 &